In [57]:
import os
import pefile
import csv
import random

<b>List of NOP-equivalent opcodes

In [58]:
NOP_EQUIVALENTS = [
    b'\x83\xC0\x00',  # ADD EAX, 0
    b'\x83\xE8\x00',  # SUB EAX, 0
    b'\x6B\xC0\x01',  # IMUL EAX, EAX, 1
    b'\x8D\x40\x00',  # LEA EAX, [EAX + 0]
    b'\x0B\xC0',      # OR EAX, EAX
    b'\x23\xC0',      # AND EAX, EAX
    b'\xD9\xD0'       # FNOP
    b'\x89\xC0'       # MOV EAX EAX
    b'\x87\xC0'       # XCHG EAX EAX
]

<b> Find and return the target section for NOP injection

In [59]:
def find_target_section(pe):
    # Select the section with the highest slack space
    max_slack = 0
    target_section = None
    section_name = ""
    
    for section in pe.sections:
        section_data = section.get_data()
        zero_count = get_slack_space(section_data)
        
        if zero_count > max_slack:
            max_slack = zero_count
            target_section = section
            section_name = section.Name.decode(errors='ignore').strip('\x00')
    
    if target_section is None:
        raise ValueError("No suitable section found for modification")
    
    return target_section, section_name

<b>Calculate the amount of slack space at the end of the section

In [60]:
def get_slack_space(section_data):
    zero_count = 0
    for byte in reversed(section_data):
        if byte == 0x00:
            zero_count += 1
        else:
            break
    return zero_count

<b> Inject random NOP-equivalent code into the slack space of a PE file

In [61]:
def inject_random_nop(file_path, nop_percentage=0.05, nop_code=None):
    pe = pefile.PE(file_path)
    target_section, section_name = find_target_section(pe)
    
    section_data = target_section.get_data()
    zero_count = get_slack_space(section_data)

    if zero_count == 0:
        raise ValueError(f"No contiguous zero bytes found in section {section_name}")

    injection_size = int(zero_count * nop_percentage)
    if injection_size == 0:
        raise ValueError(f"Injection size is zero for {nop_percentage*100}% of slack space")

    if nop_code is None:
        nop_code = random.choice(NOP_EQUIVALENTS)
    nop_code_size = len(nop_code)

    if injection_size < nop_code_size:
        raise ValueError(f"Not enough slack space in {section_name} to inject {nop_code_size} byte NOP code")

    last_free_space_offset = len(section_data) - zero_count
    file_offset = target_section.PointerToRawData + last_free_space_offset

    with open(file_path, 'rb') as f:
        original_data = f.read()

    modified_data = (
        original_data[:file_offset] +
        nop_code +
        original_data[file_offset + nop_code_size:]
    )

    return modified_data, section_name, nop_code.hex()

<b>Process executables in the input folder, injecting NOP-equivalent code and logging results

In [62]:
def process_executables(input_folder, output_folder_base, csv_path, start_percentage=0.05, step=0.10, iterations=10):
    if not os.path.exists(output_folder_base):
        os.makedirs(output_folder_base)

    # Read existing CSV entries
    processed_files = {}
    if os.path.exists(csv_path):
        with open(csv_path, mode='r') as csv_file:
            csv_reader = csv.reader(csv_file)
            next(csv_reader)  # Skip header
            for row in csv_reader:
                key = (row[0], row[1], row[2])
                processed_files[key] = (row[3], row[4])

    with open(csv_path, mode='a', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["File Name", "Modified Section", "NOP Code Inserted", "Status", "Error Message"])

        for root, _, files in os.walk(input_folder):
            for file_name in files:
                if file_name.endswith(".exe"):
                    file_path = os.path.join(root, file_name)
                    family_name = os.path.basename(root)
                    
                    # Create base folder for each percentage and family
                    for i in range(5, 101, 10):
                        nop_percentage = i / 100.0
                        output_folder = os.path.join(output_folder_base, str(i))
                        output_subfolder = os.path.join(output_folder, family_name)
                        os.makedirs(output_subfolder, exist_ok=True)

                        try:
                            # Determine the section and NOP code once
                            key = (file_name, "")
                            if key not in processed_files:
                                modified_data, section_name, nop_code_hex = inject_random_nop(file_path, 1.0)
                                processed_files[key] = (section_name, nop_code_hex)
                            else:
                                section_name, nop_code_hex = processed_files[key]

                            output_file_path = os.path.join(output_subfolder, f"modified_{i}_{file_name}")
                            
                            # Use the same NOP code for all percentages
                            modified_data, _, _ = inject_random_nop(file_path, nop_percentage, bytes.fromhex(nop_code_hex))

                            if (file_name, section_name, nop_code_hex) not in processed_files:
                                csv_writer.writerow([file_name, section_name, nop_code_hex, "Success", "None"])
                                processed_files[(file_name, section_name, nop_code_hex)] = ("Success", "None")

                            with open(output_file_path, 'wb') as f:
                                f.write(modified_data)

                            print(f"Modified {file_name} with {nop_percentage*100}% NOP code in {section_name} saved as {output_file_path}")

                        except Exception as e:
                            print(f"Error processing {file_name}: {e}")
                            csv_writer.writerow([file_name, "N/A", "N/A", "Failed", str(e)])


In [65]:
input_folder = "/media/doonu/H/Problem_Space/Dummy/"
output_folder_base = "/media/doonu/H/Problem_Space/Manipulated_Executable_NOP/"
csv_path = "/media/doonu/H/Problem_Space/DUmmy_Modified_sections_log.csv"
process_executables(input_folder, output_folder_base, csv_path)

Modified emotet14.exe with 5.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/emotet/modified_5_emotet14.exe
Modified emotet14.exe with 15.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/emotet/modified_15_emotet14.exe
Modified emotet14.exe with 25.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/emotet/modified_25_emotet14.exe
Modified emotet14.exe with 35.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/emotet/modified_35_emotet14.exe
Modified emotet14.exe with 45.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/emotet/modified_45_emotet14.exe
Modified emotet14.exe with 55.00000000000001% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/55/emotet/modified_55_emotet14.exe
Modified emotet14.exe with 65.0% NOP code in  saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/65/

Modified emotet448.exe with 15.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/emotet/modified_15_emotet448.exe
Modified emotet448.exe with 25.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/emotet/modified_25_emotet448.exe
Modified emotet448.exe with 35.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/emotet/modified_35_emotet448.exe
Modified emotet448.exe with 45.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/emotet/modified_45_emotet448.exe
Modified emotet448.exe with 55.00000000000001% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/55/emotet/modified_55_emotet448.exe
Modified emotet448.exe with 65.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/65/emotet/modified_65_emotet448.exe
Modified emotet448.exe with 75.0% NOP code in UPX1 saved as /media/doonu/H/

Modified emotet456.exe with 75.0% NOP code in .data saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/75/emotet/modified_75_emotet456.exe
Modified emotet456.exe with 85.0% NOP code in .data saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/85/emotet/modified_85_emotet456.exe
Modified emotet456.exe with 95.0% NOP code in .data saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/95/emotet/modified_95_emotet456.exe
Modified emotet222.exe with 5.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/emotet/modified_5_emotet222.exe
Modified emotet222.exe with 15.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/emotet/modified_15_emotet222.exe
Modified emotet222.exe with 25.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/emotet/modified_25_emotet222.exe
Modified emotet222.exe with 35.0% NOP code in .reloc saved as /media/doonu/H/Probl

Modified emotet130.exe with 65.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/65/emotet/modified_65_emotet130.exe
Modified emotet130.exe with 75.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/75/emotet/modified_75_emotet130.exe
Modified emotet130.exe with 85.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/85/emotet/modified_85_emotet130.exe
Modified emotet130.exe with 95.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/95/emotet/modified_95_emotet130.exe
Modified emotet35.exe with 5.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/emotet/modified_5_emotet35.exe
Modified emotet35.exe with 15.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/emotet/modified_15_emotet35.exe
Modified emotet35.exe with 25.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/

Modified emotet248.exe with 55.00000000000001% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/55/emotet/modified_55_emotet248.exe
Modified emotet248.exe with 65.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/65/emotet/modified_65_emotet248.exe
Modified emotet248.exe with 75.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/75/emotet/modified_75_emotet248.exe
Modified emotet248.exe with 85.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/85/emotet/modified_85_emotet248.exe
Modified emotet248.exe with 95.0% NOP code in .text saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/95/emotet/modified_95_emotet248.exe
Modified emotet56.exe with 5.0% NOP code in UPX1 saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/emotet/modified_5_emotet56.exe
Modified emotet56.exe with 15.0% NOP code in UPX1 saved as /media/doonu/H/P

...

Modified spyware_formbook_2020_73.exe with 5.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2020_73.exe
Modified spyware_formbook_2020_73.exe with 15.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2020_73.exe
Modified spyware_formbook_2020_73.exe with 25.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2020_73.exe
Modified spyware_formbook_2020_73.exe with 35.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2020_73.exe
Modified spyware_formbook_2020_73.exe with 45.0% NOP code in .rsrc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2020_73.exe
Modified spyware_formbook_2020_73.exe with 55.00000000000001% NOP code in .rsr

Modified spyware_formbook_2021_227.exe with 45.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_227.exe
Modified spyware_formbook_2021_227.exe with 55.00000000000001% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/55/Formbook/modified_55_spyware_formbook_2021_227.exe
Modified spyware_formbook_2021_227.exe with 65.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/65/Formbook/modified_65_spyware_formbook_2021_227.exe
Modified spyware_formbook_2021_227.exe with 75.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/75/Formbook/modified_75_spyware_formbook_2021_227.exe
Modified spyware_formbook_2021_227.exe with 85.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/85/Formbook/modified_85_spyware_formbook_2021_227.exe
Modified spyware_formbook_2021_227.exe with 95.

Modified spyware_formbook_2020_150.exe with 5.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2020_150.exe
Modified spyware_formbook_2020_150.exe with 15.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2020_150.exe
Modified spyware_formbook_2020_150.exe with 25.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2020_150.exe
Modified spyware_formbook_2020_150.exe with 35.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2020_150.exe
Modified spyware_formbook_2020_150.exe with 45.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2020_150.exe
Modified spyware_formbook_2020_150.exe with 55.00000000000001% 

Modified spyware_formbook_2021_499.exe with 5.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2021_499.exe
Modified spyware_formbook_2021_499.exe with 15.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2021_499.exe
Modified spyware_formbook_2021_499.exe with 25.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2021_499.exe
Modified spyware_formbook_2021_499.exe with 35.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2021_499.exe
Modified spyware_formbook_2021_499.exe with 45.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/45/Formbook/modified_45_spyware_formbook_2021_499.exe
Modified spyware_formbook_2021_499.exe with 55.00000000000001% 

Modified spyware_formbook_2020_97.exe with 95.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/95/Formbook/modified_95_spyware_formbook_2020_97.exe
Modified spyware_formbook_2020_265.exe with 5.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/5/Formbook/modified_5_spyware_formbook_2020_265.exe
Modified spyware_formbook_2020_265.exe with 15.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/15/Formbook/modified_15_spyware_formbook_2020_265.exe
Modified spyware_formbook_2020_265.exe with 25.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/25/Formbook/modified_25_spyware_formbook_2020_265.exe
Modified spyware_formbook_2020_265.exe with 35.0% NOP code in .reloc saved as /media/doonu/H/Problem_Space/Manipulated_Executable_NOP/35/Formbook/modified_35_spyware_formbook_2020_265.exe
Modified spyware_formbook_2020_265.exe with 45.0% NOP code in .re